In [ ]:
#Index,Symbol,Underlying Asset
#1,MIDCPNIFTY,NSE:MIDCPNIFTY-INDEX
#2,FINNIFTY,NSE:FINNIFTY-INDEX
#3,NIFTY,NSE:NIFTY50-INDEX
#4,BANKNIFTY,NSE:NIFTYBANK-INDEX
#5,NIFTYNXT50,NSE:NIFTYNXT50-INDEX

In [ ]:
!pip install fyers_apiv3

In [ ]:
client_id = "" # your clinet id
secret_key = "" # your secert key

In [ ]:
base_directory = "/content/BNF"
AT1=base_directory
jack=AT1

In [ ]:
#TICKERS MIDDLE STRING
#FOR EXAMPLE NSE:MIDCPNIFTY "24610"  12775PE
month_year = '24724'

In [ ]:
#NSE_FO.csv syntax ,EXAMPLE "24 May"
VALID_DATE = "24 Jul"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
access_token=# acces_toekn

In [ ]:
#STARTING DATE SHOULD BE 3 DAYS BACK OF THE STARTING DATE OF THE CONTRACT , TO TRACK THE TWO DAYS BACK VOLUE OF THE OPTIONS CHAINS (AND INCASE OF HOLIDAYS)
date1 = "2024-07-15"
date2 = "2024-07-24"


All CSV files in /content have been deleted.


# ADJUST OCCASIONALLY


In [ ]:
import os

# Define the directories to be created
directories = ['/content/BIN_F', '/content/LOG_F']

# Create the directories if they do not exist
for directory in directories:
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory {directory} created.")
    else:
        print(f"Directory {directory} already exists.")


Directory /content/BIN_F created.
Directory /content/LOG_F created.


In [ ]:
BIN_F='/content/BIN_F'
LOG_F= '/content/LOG_F'

In [ ]:
#INTERVAL OF REV TICKERS, FOR 10 TICKERS REV GIVE 10+1
OFFSET=11

In [ ]:
NSE_FO_PATH_DRIVE ='/content/drive/MyDrive/TCSV/NSE_FO.csv'

In [ ]:
#Downloading Underlying Asset

# Downloading DATA

In [ ]:
import csv
from datetime import datetime
import pytz
import os
import time

from fyers_apiv3 import fyersModel

# Initialize FyersModel instance
fyers = fyersModel.FyersModel(client_id=client_id, is_async=False, token=access_token, log_path="")

# Define Indian Standard Time (IST) timezone
ist = pytz.timezone('Asia/Kolkata')

# Read file names and symbols from the CSV file
filename_to_symbol = {}
with open('/content/Comp_FO.csv', newline='') as csvfile:
    reader = csv.reader(csvfile)
    next(reader)  # Skip the header
    for row in reader:
        filename_to_symbol[row[2]] = row[2]  # Assumes 'Index' is in column 0, 'File Name' in column 1, 'Symbol' in column 2

# Function to collect and write data for a date range
def collect_and_write_data(date_from, date_to, filename, symbol):
    data = {
        "symbol": symbol,
        "resolution": "1D",
        "date_format": "1",
        "range_from": date_from,
        "range_to": date_to,
        "cont_flag": "1",
    }

    try:
        response = fyers.history(data=data)
        print(symbol, date_to, response)

        filename = f'/content/DATA1/{filename}.csv'  # Use filename derived from column 1
        os.makedirs(os.path.dirname(filename), exist_ok=True)

        with open(filename, 'a', newline='') as csvfile:
            writer = csv.writer(csvfile)
            if csvfile.tell() == 0:
                headers = ['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume']
                writer.writerow(headers)

            for candle in response['candles']:
                timestamp = candle[0]
                timestamp_ist = datetime.fromtimestamp(timestamp, ist).strftime('%Y-%m-%d %H:%M:%S')
                writer.writerow([timestamp_ist] + candle[1:])
    except Exception as e:
        print(f"Error collecting data for {symbol}: {e}")

# Example date ranges and symbol processing

for filename, symbol in filename_to_symbol.items():
    collect_and_write_data(date1, date2, filename, symbol)

print("Data collection completed.")


NSE:NIFTYBANK-INDEX 2024-07-24 {'candles': [[1721001600, 52330.05, 52662.25, 52154, 52455.9, 0], [1721088000, 52466.7, 52619.05, 52331.6, 52396.8, 0], [1721260800, 52215.05, 52782.75, 52168.65, 52620.7, 0], [1721347200, 52531.55, 52586.75, 52146.3, 52265.6, 0], [1721606400, 52145.6, 52427, 51874.55, 52280.4, 0], [1721692800, 52511, 52547.55, 51342.65, 51778.3, 0], [1721779200, 51657.65, 51944.65, 50784.25, 51317, 0]], 'code': 200, 'message': '', 's': 'ok'}
Data collection completed.


In [ ]:
#Lowest Low and Highest High

In [ ]:
import csv
import os

# Function to calculate highest and lowest values
def get_price_stats(file_path):
    try:
        with open(file_path, 'r', newline='') as csvfile:
            # Sniff delimiter and reset read position
            dialect = csv.Sniffer().sniff(csvfile.read(1024))
            csvfile.seek(0)
            reader = csv.DictReader(csvfile, delimiter=dialect.delimiter)

            # Verify required columns are in headers
            if 'High' not in reader.fieldnames or 'Low' not in reader.fieldnames:
                print(f"Columns 'High' or 'Low' not found in {file_path}. Headers found: {reader.fieldnames}")
                return None, None

            lowest_low = None
            highest_high = None
            for row in reader:
                try:
                    high_price = float(row['High'])
                    low_price = float(row['Low'])

                    # Track lowest and highest
                    if highest_high is None or high_price > highest_high:
                        highest_high = high_price
                    if lowest_low is None or low_price < lowest_low:
                        lowest_low = low_price
                except ValueError as e:
                    print(f"Error parsing values in {file_path}: {e}")

            return lowest_low, highest_high
    except Exception as e:
        print(f"Error reading file {file_path}: {e}")
        return None, None

# Main CSV path for the input data
path_to_your_initial_csv = 'Comp_FO.csv'

# Path to save the new CSV file
path_to_new_csv = f'{LOG_F}/Monthly_HL.csv'

# Read the existing data and append the new columns
updated_rows = []
with open(path_to_your_initial_csv, 'r', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    fieldnames = reader.fieldnames + ['Lowest Low', 'Highest High']
    for row in reader:
        symbol = row['Underlying Asset']
        file_path = os.path.join('/content/DATA1', f'{symbol}.csv')
        lowest_low, highest_high = get_price_stats(file_path)
        row['Lowest Low'] = lowest_low
        row['Highest High'] = highest_high
        updated_rows.append(row)

# Write the updated data to a new CSV file
with open(path_to_new_csv, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(updated_rows)

print(f"Updated data saved to {path_to_new_csv}.")


Updated data saved to /content/LOG_F/Monthly_HL.csv.


In [ ]:
#ACCESSING NSE_FO.csv

In [ ]:
import pandas as pd
# Load datasets from CSV files
broker_data = pd.read_csv('/content/drive/MyDrive/TCSV/NSE_FO.csv', header=None)  # Adjust the path and header parameter as needed
comp_data = pd.read_csv(f'{LOG_F}/Monthly_HL.csv')  # Adjust the path as needed
# Adjust the path as needed


In [ ]:
#ADD VALID DATE AND MONTH

In [ ]:
import pandas as pd

# Constants
 # Make sure the date format exactly matches the format in the descriptions


broker_data = pd.read_csv('/content/drive/MyDrive/TCSV/NSE_FO.csv', header=None, names=[
    'id', 'description', 'unknown2', 'lot_size', 'unknown4', 'unknown5',
    'unknown6', 'market_times', 'date', 'formatted_symbol', 'unknown10',
    'unknown11', 'ref_id', 'symbol', 'contract_size', 'strike',
    'option_type', 'contract_code', 'unknown17', 'unknown18', 'unknown19'
])

# Normalize broker data
broker_data['symbol'] = broker_data['description'].str.extract(r'([A-Za-z&\.\-]+)')
broker_data['option_type'] = broker_data['description'].str.extract(r'(\bCE\b|\bPE\b)')
broker_data['strike'] = pd.to_numeric(broker_data['strike'], errors='coerce')
broker_data['lot_size'] = pd.to_numeric(broker_data['lot_size'], errors='coerce')

# Filter data by the valid date within the description
broker_data = broker_data[broker_data['description'].str.contains(VALID_DATE, na=False)]

# Debugging output to ensure data is present
print("Filtered broker data after date filtering:", broker_data.head())

# Remove rows where necessary fields could not be converted and drop duplicates
broker_data.dropna(subset=['strike', 'lot_size', 'option_type'], inplace=True)
broker_data.drop_duplicates(subset=['symbol', 'strike', 'option_type'], inplace=True)

# Normalize symbols in both datasets
comp_data['Symbol'] = comp_data['Symbol'].str.upper()
broker_data['symbol'] = broker_data['symbol'].str.upper()

# Function to find comprehensive strike ranges
def find_comprehensive_strike_ranges(symbol, low, high, option_type):
    filtered = broker_data[(broker_data['symbol'] == symbol) & (broker_data['option_type'] == option_type)]
    filtered = filtered.sort_values(by='strike')

    strikes = filtered['strike'].unique()
    below_strikes = [strike for strike in strikes if strike < low][-OFFSET:]
    between_strikes = [strike for strike in strikes if low <= strike <= high]
    above_strikes = [strike for strike in strikes if strike > high][:OFFSET]

    return below_strikes + between_strikes + above_strikes

# Apply the function to each row in Comp_FO.csv and store the results in new columns
comp_data['CE Strikes Range'] = comp_data.apply(
    lambda row: find_comprehensive_strike_ranges(row['Symbol'], row['Lowest Low'], row['Highest High'], 'CE'), axis=1)
comp_data['PE Strikes Range'] = comp_data.apply(
    lambda row: find_comprehensive_strike_ranges(row['Symbol'], row['Lowest Low'], row['Highest High'], 'PE'), axis=1)

# Lot Size Mapping: Use the first valid lot size per symbol, assuming consistency
lot_size_mapping = broker_data.groupby('symbol')['lot_size'].first()
comp_data['Lot Size'] = comp_data['Symbol'].map(lot_size_mapping)

# Save the updated DataFrame to a new CSV file
comp_data.to_csv(f'{LOG_F}/MONTHLY_REV_HL_TICKERS.csv', index=False)
print("Updated COMP_FO.csv with full CE and PE strike ranges and lot sizes has been saved.")


Filtered broker data after date filtering:                 id                    description  unknown2  lot_size  \
0  101124070835034  MIDCPNIFTY 24 Jul 08 14575 CE        14        75   
1  101124070835035  MIDCPNIFTY 24 Jul 08 14575 PE        14        75   
2  101124070835036  MIDCPNIFTY 24 Jul 08 14625 CE        14        75   
3  101124070835037  MIDCPNIFTY 24 Jul 08 14625 PE        14        75   
4  101124070835038  MIDCPNIFTY 24 Jul 08 14650 CE        14        75   

   unknown4  unknown5              unknown6 market_times        date  \
0      0.05       NaN  0915-1530|1815-1915:   2024-07-05  1720432800   
1      0.05       NaN  0915-1530|1815-1915:   2024-07-05  1720432800   
2      0.05       NaN  0915-1530|1815-1915:   2024-07-05  1720432800   
3      0.05       NaN  0915-1530|1815-1915:   2024-07-05  1720432800   
4      0.05       NaN  0915-1530|1815-1915:   2024-07-05  1720432800   

             formatted_symbol  ...  unknown11  ref_id      symbol  \
0  NSE:MIDCPNIFT

<ipython-input-18-035720e8f0d0>:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  broker_data.dropna(subset=['strike', 'lot_size', 'option_type'], inplace=True)
<ipython-input-18-035720e8f0d0>:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  broker_data.drop_duplicates(subset=['symbol', 'strike', 'option_type'], inplace=True)
<ipython-input-18-035720e8f0d0>:32: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#retu

In [ ]:
#LIST OF LIST OF LIST

In [ ]:
#APPLY THE SYNTAX OF 24XXXX

In [ ]:
import pandas as pd
import ast  # To safely evaluate string literals as Python expressions

# Load data from CSV
options_data = pd.read_csv(f'{LOG_F}//MONTHLY_REV_HL_TICKERS.csv')

# Clean the 'Symbol' column to remove any unwanted whitespace characters
options_data['Symbol'] = options_data['Symbol'].str.replace(r'\s+', '', regex=True)

# Global variable for month and year


# Function to format strike prices correctly
def format_strike_price(strike):
    if isinstance(strike, float) and strike.is_integer():
        return f"{int(strike)}"  # Convert to integer if the number is whole
    else:
        formatted_strike = f"{strike:.2f}"  # Format with two decimals
        # Remove unnecessary trailing zeros and the point if not needed
        return formatted_strike.rstrip('0').rstrip('.') if '.' in formatted_strike else formatted_strike

# Function to generate option tickers as nested lists
def generate_option_tickers(row):
    # Convert string representations of lists into actual lists
    ce_strikes = ast.literal_eval(row['CE Strikes Range'])
    pe_strikes = ast.literal_eval(row['PE Strikes Range'])

    # Format the tickers with symbol, date, and option type
    ce_tickers = [f"NSE:{row['Symbol']}{month_year}{format_strike_price(strike)}CE" for strike in ce_strikes]
    pe_tickers = [f"NSE:{row['Symbol']}{month_year}{format_strike_price(strike)}PE" for strike in pe_strikes]

    # Return nested list [list for CE, list for PE]
    return [ce_tickers, pe_tickers]

# Apply the function to each row in the DataFrame and collect results in 'symbols'
symbols = options_data.apply(generate_option_tickers, axis=1).tolist()

# Now 'symbols' contains all the nested lists structured by symbol and option type
print(symbols)



[[['NSE:BANKNIFTY2472449700CE', 'NSE:BANKNIFTY2472449800CE', 'NSE:BANKNIFTY2472449900CE', 'NSE:BANKNIFTY2472450000CE', 'NSE:BANKNIFTY2472450100CE', 'NSE:BANKNIFTY2472450200CE', 'NSE:BANKNIFTY2472450300CE', 'NSE:BANKNIFTY2472450400CE', 'NSE:BANKNIFTY2472450500CE', 'NSE:BANKNIFTY2472450600CE', 'NSE:BANKNIFTY2472450700CE', 'NSE:BANKNIFTY2472450800CE', 'NSE:BANKNIFTY2472450900CE', 'NSE:BANKNIFTY2472451000CE', 'NSE:BANKNIFTY2472451100CE', 'NSE:BANKNIFTY2472451200CE', 'NSE:BANKNIFTY2472451300CE', 'NSE:BANKNIFTY2472451400CE', 'NSE:BANKNIFTY2472451500CE', 'NSE:BANKNIFTY2472451600CE', 'NSE:BANKNIFTY2472451700CE', 'NSE:BANKNIFTY2472451800CE', 'NSE:BANKNIFTY2472451900CE', 'NSE:BANKNIFTY2472452000CE', 'NSE:BANKNIFTY2472452100CE', 'NSE:BANKNIFTY2472452200CE', 'NSE:BANKNIFTY2472452300CE', 'NSE:BANKNIFTY2472452400CE', 'NSE:BANKNIFTY2472452500CE', 'NSE:BANKNIFTY2472452600CE', 'NSE:BANKNIFTY2472452700CE', 'NSE:BANKNIFTY2472452800CE', 'NSE:BANKNIFTY2472452900CE', 'NSE:BANKNIFTY2472453000CE', 'NSE:BANKNI

In [ ]:
# Flatten the list of lists of lists to just a list of tickers
flat_list = [ticker for sublist in symbols for tickertype in sublist for ticker in tickertype]

# Count the total number of tickers
total_tickers = len(flat_list)

print("Total number of option tickers:", total_tickers)


Total number of option tickers: 84


In [ ]:
print(symbols[0])

[['NSE:BANKNIFTY2472449700CE', 'NSE:BANKNIFTY2472449800CE', 'NSE:BANKNIFTY2472449900CE', 'NSE:BANKNIFTY2472450000CE', 'NSE:BANKNIFTY2472450100CE', 'NSE:BANKNIFTY2472450200CE', 'NSE:BANKNIFTY2472450300CE', 'NSE:BANKNIFTY2472450400CE', 'NSE:BANKNIFTY2472450500CE', 'NSE:BANKNIFTY2472450600CE', 'NSE:BANKNIFTY2472450700CE', 'NSE:BANKNIFTY2472450800CE', 'NSE:BANKNIFTY2472450900CE', 'NSE:BANKNIFTY2472451000CE', 'NSE:BANKNIFTY2472451100CE', 'NSE:BANKNIFTY2472451200CE', 'NSE:BANKNIFTY2472451300CE', 'NSE:BANKNIFTY2472451400CE', 'NSE:BANKNIFTY2472451500CE', 'NSE:BANKNIFTY2472451600CE', 'NSE:BANKNIFTY2472451700CE', 'NSE:BANKNIFTY2472451800CE', 'NSE:BANKNIFTY2472451900CE', 'NSE:BANKNIFTY2472452000CE', 'NSE:BANKNIFTY2472452100CE', 'NSE:BANKNIFTY2472452200CE', 'NSE:BANKNIFTY2472452300CE', 'NSE:BANKNIFTY2472452400CE', 'NSE:BANKNIFTY2472452500CE', 'NSE:BANKNIFTY2472452600CE', 'NSE:BANKNIFTY2472452700CE', 'NSE:BANKNIFTY2472452800CE', 'NSE:BANKNIFTY2472452900CE', 'NSE:BANKNIFTY2472453000CE', 'NSE:BANKNIF

In [ ]:
#REFINE THE SYMOLSAS PER THE VALIDTY
symbols=symbols[0]

In [ ]:
print(symbols)

[['NSE:BANKNIFTY2472449700CE', 'NSE:BANKNIFTY2472449800CE', 'NSE:BANKNIFTY2472449900CE', 'NSE:BANKNIFTY2472450000CE', 'NSE:BANKNIFTY2472450100CE', 'NSE:BANKNIFTY2472450200CE', 'NSE:BANKNIFTY2472450300CE', 'NSE:BANKNIFTY2472450400CE', 'NSE:BANKNIFTY2472450500CE', 'NSE:BANKNIFTY2472450600CE', 'NSE:BANKNIFTY2472450700CE', 'NSE:BANKNIFTY2472450800CE', 'NSE:BANKNIFTY2472450900CE', 'NSE:BANKNIFTY2472451000CE', 'NSE:BANKNIFTY2472451100CE', 'NSE:BANKNIFTY2472451200CE', 'NSE:BANKNIFTY2472451300CE', 'NSE:BANKNIFTY2472451400CE', 'NSE:BANKNIFTY2472451500CE', 'NSE:BANKNIFTY2472451600CE', 'NSE:BANKNIFTY2472451700CE', 'NSE:BANKNIFTY2472451800CE', 'NSE:BANKNIFTY2472451900CE', 'NSE:BANKNIFTY2472452000CE', 'NSE:BANKNIFTY2472452100CE', 'NSE:BANKNIFTY2472452200CE', 'NSE:BANKNIFTY2472452300CE', 'NSE:BANKNIFTY2472452400CE', 'NSE:BANKNIFTY2472452500CE', 'NSE:BANKNIFTY2472452600CE', 'NSE:BANKNIFTY2472452700CE', 'NSE:BANKNIFTY2472452800CE', 'NSE:BANKNIFTY2472452900CE', 'NSE:BANKNIFTY2472453000CE', 'NSE:BANKNIF

In [ ]:
#GOLD

In [ ]:
#GOLD

In [ ]:
#GOLD

In [ ]:

import pandas as pd
import csv
from datetime import datetime
import pytz
import os
import time
from fyers_apiv3 import fyersModel

# API initialization
fyers = fyersModel.FyersModel(client_id=client_id, is_async=False, token=access_token, log_path="")

# Define timezone for India
ist = pytz.timezone('Asia/Kolkata')

# Function to process and save data
def collect_and_write_data(date_from, date_to, symbol, base_directory, resolution):
    # Adjust resolution string for directory naming
    res_dir = resolution if resolution != "1" else "1M"

    data_request = {
        "symbol": symbol,
        "resolution": resolution,
        "date_format": "1",
        "range_from": date_from,
        "range_to": date_to,
        "cont_flag": "1",
        "oi_flag": "1"  # Enable Open Interest data
    }
    response = fyers.history(data=data_request)
    print(f"Response for {symbol} at resolution {resolution}")  # Print the API response for debugging

    # Extract formatted month and day from date2
    formatted_month = datetime.strptime(date_to, "%Y-%m-%d").strftime("%b").upper()  # Month in uppercase
    day_of_month = datetime.strptime(date_to, "%Y-%m-%d").day  # Day of the month

    option_type = 'CE' if symbol.endswith('CE') else 'PE'
    directory = f"{base_directory}/{formatted_month}_{day_of_month}/{formatted_month}_{res_dir}_{day_of_month}/{option_type}"

    # Ensure the directory includes the resolution
    os.makedirs(directory, exist_ok=True)
    filename = f"{directory}/{symbol}.csv"

    with open(filename, 'a', newline='') as csvfile:
        writer = csv.writer(csvfile)
        if csvfile.tell() == 0:  # Only write headers if file is empty
            writer.writerow(['Timestamp', 'Open', 'High', 'Low', 'Close', 'Volume', 'OI'])
        for candle in response.get('candles', []):
            timestamp = datetime.fromtimestamp(candle[0], ist).strftime('%Y-%m-%d %H:%M:%S')
            writer.writerow([timestamp] + candle[1:])

# Track total process time
start_time = time.time()

# Process each symbol and determine file path
resolutions = ['5S']#, '15S', '30S', '1']

for resolution in resolutions:
    for symbol_group in symbols:  # symbols is a list of lists
        for symbol in symbol_group:  # Iterate through each symbol in each sublist
            collect_and_write_data(date1, date2, symbol, base_directory, resolution)

end_time = time.time()
total_duration = end_time - start_time
print(f"Data collection and writing completed successfully. Total execution time: {total_duration}")


# HL(REV HL , TICKER)


In [ ]:
#Closig price for eveery day
import pandas as pd
import os

# Path to the directory containing the index files
data_directory = '/content/DATA1/'

# Load Index_FO data
index_fo_path = 'Comp_FO.csv'
index_fo_data = pd.read_csv(index_fo_path)

# Assuming 'Underlying Asset' column provides the filenames directly related to symbol CSVs
# Generate full file paths and check existence
index_fo_data['File Path'] = index_fo_data['Underlying Asset'].apply(lambda x: os.path.join(data_directory, f"{x}.csv"))

# Prepare to append data
all_dates = set()

# Iterate through each row in Index_FO to find corresponding index data and append closing prices
for idx, row in index_fo_data.iterrows():
    file_path = row['File Path']
    if os.path.exists(file_path):
        # Load the data from file
        data = pd.read_csv(file_path)
        data['Timestamp'] = pd.to_datetime(data['Timestamp']).dt.date  # Ensure date format

        # Extract and append unique dates
        unique_dates = data['Timestamp'].unique()
        all_dates.update(unique_dates)

        # Append closing prices
        for date in unique_dates:
            # Ensure the column for the date
            if date not in index_fo_data.columns:
                index_fo_data[date] = pd.NA  # Initialize with 'pd.NA' for missing data

            # Get the closing price for the date
            close_price = data.loc[data['Timestamp'] == date, 'Close'].iloc[0]  # Ensure correct extraction
            index_fo_data.at[idx, date] = close_price

# Remove the 'File Path' column from the DataFrame
index_fo_data.drop('File Path', axis=1, inplace=True)

# Save the modified Index_FO data
output_path = f'{BIN_F}/PREV_CLOSING.csv'
index_fo_data.to_csv(output_path, index=False)

# Print to confirm output
print(index_fo_data.head())


   Index     Symbol     Underlying Asset 2024-07-15 2024-07-16 2024-07-18  \
0     15  BANKNIFTY  NSE:NIFTYBANK-INDEX    52455.9    52396.8    52620.7   

  2024-07-19 2024-07-22 2024-07-23 2024-07-24  
0    52265.6    52280.4    51778.3    51317.0  


In [ ]:
#Z
import csv
import re

# Open the input CSV file
with open('/content/drive/MyDrive/TCSV/NSE_FO.csv', newline='') as csvfile:
    # Create a CSV reader object
    reader = csv.reader(csvfile)

    # Open the output CSV file to write the filtered data
    with open(f'{BIN_F}/broker.csv', 'w', newline='') as output_csvfile:
        # Create a CSV writer object
        writer = csv.writer(output_csvfile)

        # Iterate through each row in the input CSV file
        for row in reader:
            # Check if the first column contains "24 May" using regex
            if re.search(VALID_DATE, row[1]):
                # Append the first column and corresponding 15th column to the output CSV file
                writer.writerow([row[1], row[15]])


In [ ]:
import pandas as pd
df=pd.read_csv(f'{BIN_F}/broker.csv')
df.head(10)
df.tail(10)

,MIDCPNIFTY 24 Jul 08 14575 CE,14575.0
34938,BANKNIFTY 24 Jul 31 60100 CE,60100.0
34939,BANKNIFTY 24 Jul 31 60100 PE,60100.0
34940,BANKNIFTY 24 Jul 31 60200 CE,60200.0
34941,BANKNIFTY 24 Jul 31 60200 PE,60200.0
34942,BANKNIFTY 24 Jul 31 60300 CE,60300.0
34943,BANKNIFTY 24 Jul 31 60300 PE,60300.0
34944,BANKNIFTY 24 Jul 31 60400 CE,60400.0
34945,BANKNIFTY 24 Jul 31 60400 PE,60400.0
34946,BANKNIFTY 24 Jul 31 74000 CE,74000.0
34947,BANKNIFTY 24 Jul 31 74000 PE,74000.0


In [ ]:
import pandas as pd
from collections import defaultdict
import os

# Load the list of companies and their files
comp_fo_path = 'Comp_FO.csv'
comp_fo_data = pd.read_csv(comp_fo_path)

# Directory where individual ticker CSVs are stored
base_directory = '/content/DATA1/'

# Dictionary to hold data for each symbol
symbol_data = defaultdict(lambda: defaultdict(list))

# Process each company's file
for index, row in comp_fo_data.iterrows():
    symbol = row['Underlying Asset']
    file_path = os.path.join(base_directory, symbol + '.csv')

    try:
        # Read the company's data file
        data = pd.read_csv(file_path)
        data['Timestamp'] = pd.to_datetime(data['Timestamp']).dt.date  # Normalize to date only

        # Aggregate to find daily high and low
        daily_highs = data.groupby('Timestamp')['High'].max()
        daily_lows = data.groupby('Timestamp')['Low'].min()

        # Store the results in the dictionary as a list [low, high]
        for date in daily_highs.index:
            date_key = f"{date} HL"  # Format the date key as 'YYYY-MM-DD HL'
            symbol_data[symbol][date_key] = [daily_lows[date], daily_highs[date]]
    except FileNotFoundError:
        print(f"File not found for symbol: {symbol}")
    except Exception as e:
        print(f"Error processing file {file_path}: {e}")

# Update the Comp_FO.csv with the new columns
new_rows = []
for index, row in comp_fo_data.iterrows():
    symbol = row['Underlying Asset']
    # Append the high and low values to each row
    for date_key, values in symbol_data[symbol].items():
        row[date_key] = values
    new_rows.append(row)

# Convert list of rows to DataFrame
updated_comp_fo = pd.DataFrame(new_rows)

# Save the updated DataFrame
updated_path = f'{LOG_F}/CURR_DAY_HL.csv'
updated_comp_fo.to_csv(updated_path, index=False)

print(f"Updated data saved to {updated_path}.")


Updated data saved to /content/LOG_F/CURR_DAY_HL.csv.


In [ ]:
import pandas as pd
import re
import ast  # Import ast to safely evaluate strings as lists

# Load the HL.csv data
hl_data = pd.read_csv(f'{LOG_F}/CURR_DAY_HL.csv')


# Load broker data from the CSV
broker_data = pd.read_csv('/content/drive/MyDrive/TCSV/NSE_FO.csv', header=None)
broker_data.columns = [
    'ID', 'Description', 'Unknown1', 'Lot Size', 'Unknown2', 'Unknown3', 'Trading Window', 'Date',
    'Timestamp', 'Contract Name', 'Unknown4', 'Unknown5', 'Unknown6', 'Symbol', 'Unknown7', 'Strike Price',
    'Option Type', 'Unknown8', 'Unknown9', 'Unknown10', 'Unknown11'
]

# Convert 'Strike Price' to numeric
broker_data['Strike Price'] = pd.to_numeric(broker_data['Strike Price'], errors='coerce')

# Function to find the closest strikes within the high-low range for each option type, extending two below and two above the range
def find_strikes_within_range(symbol, high_low):
    high, low = ast.literal_eval(high_low)[1], ast.literal_eval(high_low)[0]
    results = {'CE': [], 'PE': []}

    for option_type in ['CE', 'PE']:
        symbol_strikes = broker_data[(broker_data['Symbol'] == symbol) & (broker_data['Option Type'] == option_type)]
        valid_strikes = symbol_strikes[symbol_strikes['Strike Price'].between(low, high)].sort_values(by='Strike Price')

        if not valid_strikes.empty:
            # Extend the search range to include two strikes below the lowest and two above the highest
            sorted_strikes = symbol_strikes.sort_values(by='Strike Price').drop_duplicates(subset='Strike Price').reset_index(drop=True)
            low_idx = sorted_strikes.index[sorted_strikes['Strike Price'] == valid_strikes['Strike Price'].min()].tolist()[0]
            high_idx = sorted_strikes.index[sorted_strikes['Strike Price'] == valid_strikes['Strike Price'].max()].tolist()[0]
            start_idx = max(low_idx - OFFSET, 0)
            end_idx = min(high_idx + OFFSET, len(sorted_strikes) - 1)
            extended_strikes = sorted_strikes.iloc[start_idx:end_idx + 1]['Strike Price'].tolist()
            results[option_type] = extended_strikes

    return [results['CE'], results['PE']]

# Regex to match columns that end with "HL"
hl_column_regex = re.compile(r'.*HL$')

# Process each symbol for each date in HL.csv
hl_columns = [col for col in hl_data.columns if hl_column_regex.search(col)]
for date_col in hl_columns:
    column_name = f'ATM Strike on {date_col} ATM Info'
    hl_data[column_name] = hl_data.apply(
        lambda row: find_strikes_within_range(row['Symbol'], row[date_col]), axis=1)

# Save the updated DataFrame
updated_path = f'{LOG_F}/EVERYDAY_HL+REV_TICKERS.csv'
hl_data.to_csv(updated_path, index=False)

# Display the first few rows to verify the results
print(hl_data.head())


   Index     Symbol     Underlying Asset        2024-07-15 HL  \
0     15  BANKNIFTY  NSE:NIFTYBANK-INDEX  [52154.0, 52662.25]   

         2024-07-16 HL         2024-07-18 HL        2024-07-19 HL  \
0  [52331.6, 52619.05]  [52168.65, 52782.75]  [52146.3, 52586.75]   

         2024-07-22 HL         2024-07-23 HL         2024-07-24 HL  \
0  [51874.55, 52427.0]  [51342.65, 52547.55]  [50784.25, 51944.65]   

                ATM Strike on 2024-07-15 HL ATM Info  \
0  [[51100.0, 51200.0, 51300.0, 51400.0, 51500.0,...   

                ATM Strike on 2024-07-16 HL ATM Info  \
0  [[51300.0, 51400.0, 51500.0, 51600.0, 51700.0,...   

                ATM Strike on 2024-07-18 HL ATM Info  \
0  [[51100.0, 51200.0, 51300.0, 51400.0, 51500.0,...   

                ATM Strike on 2024-07-19 HL ATM Info  \
0  [[51100.0, 51200.0, 51300.0, 51400.0, 51500.0,...   

                ATM Strike on 2024-07-22 HL ATM Info  \
0  [[50800.0, 50900.0, 51000.0, 51100.0, 51200.0,...   

                ATM Stri

In [ ]:
print(base_directory)

/content/DATA1/


In [ ]:
print(base_directory)

/content/DATA1/


# ATM BASED ON PREV DAY CLOSE (COMP)

In [ ]:
#Closig price for eveery day
import pandas as pd
import os

# Path to the directory containing the index files
data_directory = '/content/DATA1/'

# Load Index_FO data
index_fo_path = 'Comp_FO.csv'
index_fo_data = pd.read_csv(index_fo_path)

# Assuming 'Underlying Asset' column provides the filenames directly related to symbol CSVs
# Generate full file paths and check existence
index_fo_data['File Path'] = index_fo_data['Underlying Asset'].apply(lambda x: os.path.join(data_directory, f"{x}.csv"))

# Prepare to append data
all_dates = set()

# Iterate through each row in Index_FO to find corresponding index data and append closing prices
for idx, row in index_fo_data.iterrows():
    file_path = row['File Path']
    if os.path.exists(file_path):
        # Load the data from file
        data = pd.read_csv(file_path)
        data['Timestamp'] = pd.to_datetime(data['Timestamp']).dt.date  # Ensure date format

        # Extract and append unique dates
        unique_dates = data['Timestamp'].unique()
        all_dates.update(unique_dates)

        # Append closing prices
        for date in unique_dates:
            # Ensure the column for the date
            if date not in index_fo_data.columns:
                index_fo_data[date] = pd.NA  # Initialize with 'pd.NA' for missing data

            # Get the closing price for the date
            close_price = data.loc[data['Timestamp'] == date, 'Close'].iloc[0]  # Ensure correct extraction
            index_fo_data.at[idx, date] = close_price

# Remove the 'File Path' column from the DataFrame
index_fo_data.drop('File Path', axis=1, inplace=True)

# Save the modified Index_FO data
output_path = f'{BIN_F}/CLOSING_PRICE.csv'
index_fo_data.to_csv(output_path, index=False)

# Print to confirm output
print(index_fo_data.head())


   Index     Symbol     Underlying Asset 2024-07-15 2024-07-16 2024-07-18  \
0     15  BANKNIFTY  NSE:NIFTYBANK-INDEX    52455.9    52396.8    52620.7   

  2024-07-19 2024-07-22 2024-07-23 2024-07-24  
0    52265.6    52280.4    51778.3    51317.0  


In [ ]:
# Load daily prices data and print the first few rows to inspect
import pandas as pd
daily_prices_path = f'{BIN_F}/CLOSING_PRICE.csv'
daily_prices_data = pd.read_csv(daily_prices_path)
print("Daily Prices Data Sample:")
print(daily_prices_data.head())

# Load broker data, set column names, and print the first few rows to inspect
broker_data_path = NSE_FO_PATH_DRIVE
broker_data = pd.read_csv(broker_data_path, header=None)
column_names = [
    'ID', 'Description', 'Unknown1', 'Lot Size', 'Unknown2', 'Unknown3', 'Trading Window',
    'Date', 'Timestamp', 'Code', 'Unknown4', 'Unknown5', 'Unknown6', 'Index',
    'Unknown7', 'Strike Price', 'Option Type', 'Unknown8', 'Unknown9', 'Unknown10', 'Extra Column'
]
broker_data.columns = column_names
print("Broker Data Sample:")
print(broker_data.head())


Daily Prices Data Sample:
   Index     Symbol     Underlying Asset  2024-07-15  2024-07-16  2024-07-18  \
0     15  BANKNIFTY  NSE:NIFTYBANK-INDEX     52455.9     52396.8     52620.7   

   2024-07-19  2024-07-22  2024-07-23  2024-07-24  
0     52265.6     52280.4     51778.3     51317.0  
Broker Data Sample:
                ID                    Description  Unknown1  Lot Size  \
0  101124070835034  MIDCPNIFTY 24 Jul 08 14575 CE        14        75   
1  101124070835035  MIDCPNIFTY 24 Jul 08 14575 PE        14        75   
2  101124070835036  MIDCPNIFTY 24 Jul 08 14625 CE        14        75   
3  101124070835037  MIDCPNIFTY 24 Jul 08 14625 PE        14        75   
4  101124070835038  MIDCPNIFTY 24 Jul 08 14650 CE        14        75   

   Unknown2  Unknown3        Trading Window        Date   Timestamp  \
0      0.05       NaN  0915-1530|1815-1915:  2024-07-05  1720432800   
1      0.05       NaN  0915-1530|1815-1915:  2024-07-05  1720432800   
2      0.05       NaN  0915-1530|1815

In [ ]:
# Load daily prices data and print the first few rows to inspect
import pandas as pd
daily_prices_path = f'{BIN_F}/CLOSING_PRICE.csv'
daily_prices_data = pd.read_csv(daily_prices_path)
print("Daily Prices Data Sample:")
print(daily_prices_data.head())

# Load broker data, set column names, and print the first few rows to inspect
broker_data_path = NSE_FO_PATH_DRIVE
broker_data = pd.read_csv(broker_data_path, header=None)
column_names = [
    'ID', 'Description', 'Unknown1', 'Lot Size', 'Unknown2', 'Unknown3', 'Trading Window',
    'Date', 'Timestamp', 'Code', 'Unknown4', 'Unknown5', 'Unknown6', 'Index',
    'Unknown7', 'Strike Price', 'Option Type', 'Unknown8', 'Unknown9', 'Unknown10', 'Extra Column'
]
broker_data.columns = column_names
print("Broker Data Sample:")
print(broker_data.head())


Daily Prices Data Sample:
   Index     Symbol     Underlying Asset  2024-07-15  2024-07-16  2024-07-18  \
0     15  BANKNIFTY  NSE:NIFTYBANK-INDEX     52455.9     52396.8     52620.7   

   2024-07-19  2024-07-22  2024-07-23  2024-07-24  
0     52265.6     52280.4     51778.3     51317.0  
Broker Data Sample:
                ID                    Description  Unknown1  Lot Size  \
0  101124070835034  MIDCPNIFTY 24 Jul 08 14575 CE        14        75   
1  101124070835035  MIDCPNIFTY 24 Jul 08 14575 PE        14        75   
2  101124070835036  MIDCPNIFTY 24 Jul 08 14625 CE        14        75   
3  101124070835037  MIDCPNIFTY 24 Jul 08 14625 PE        14        75   
4  101124070835038  MIDCPNIFTY 24 Jul 08 14650 CE        14        75   

   Unknown2  Unknown3        Trading Window        Date   Timestamp  \
0      0.05       NaN  0915-1530|1815-1915:  2024-07-05  1720432800   
1      0.05       NaN  0915-1530|1815-1915:  2024-07-05  1720432800   
2      0.05       NaN  0915-1530|1815

In [ ]:
import csv
import re

# Global variable for the search pattern

# Open the input CSV file
with open(NSE_FO_PATH_DRIVE, newline='') as csvfile:
    # Create a CSV reader object
    reader = csv.reader(csvfile)

    # Open the output CSV file to write the filtered data
    with open(f'{BIN_F}/broker.csv', 'w', newline='') as output_csvfile:
        # Create a CSV writer object
        writer = csv.writer(output_csvfile)

        # Iterate through each row in the input CSV file
        for row in reader:
            # Check if the first column contains the search pattern using regex
            if re.search(VALID_DATE, row[1]):
                # Append the first column and corresponding 15th column to the output CSV file
                writer.writerow([row[1], row[15]])


In [ ]:
import pandas as pd
df=pd.read_csv(f'{BIN_F}/broker.csv')
df.head(10)
df.tail(10)

,MIDCPNIFTY 24 Jul 08 14575 CE,14575.0
34938,BANKNIFTY 24 Jul 31 60100 CE,60100.0
34939,BANKNIFTY 24 Jul 31 60100 PE,60100.0
34940,BANKNIFTY 24 Jul 31 60200 CE,60200.0
34941,BANKNIFTY 24 Jul 31 60200 PE,60200.0
34942,BANKNIFTY 24 Jul 31 60300 CE,60300.0
34943,BANKNIFTY 24 Jul 31 60300 PE,60300.0
34944,BANKNIFTY 24 Jul 31 60400 CE,60400.0
34945,BANKNIFTY 24 Jul 31 60400 PE,60400.0
34946,BANKNIFTY 24 Jul 31 74000 CE,74000.0
34947,BANKNIFTY 24 Jul 31 74000 PE,74000.0


In [ ]:
import pandas as pd
import numpy as np

# Load the data from the specified CSV file
data_at = pd.read_csv(f'{BIN_F}/CLOSING_PRICE.csv')
data_broker = pd.read_csv(f'{BIN_F}/broker.csv', header=None, names=['Description', 'Strike'])

# Ensure the 'Strike' column is treated as float
data_broker['Strike'] = pd.to_numeric(data_broker['Strike'], errors='coerce')
data_broker.dropna(subset=['Strike'], inplace=True)  # Drop any rows where 'Strike' could not be converted

# Initialize counters
missing_count = 0
processed_count = 0
no_strikes_count = 0

# Function to extract the symbol and find the closest strike
def find_nearest_strike(symbol, price, date, index):
    global missing_count, processed_count, no_strikes_count
    if pd.isna(price):
        missing_count += 1
        print(f"Missing price for {symbol} on {date}, row index: {index}")
        return np.nan  # Return NaN immediately if the price is missing
    else:
        price = float(price)  # Ensure price is a float to perform arithmetic operations
        filtered_data = data_broker[data_broker['Description'].str.contains(symbol, na=False)]
        symbol_strikes = filtered_data['Strike'].values
        if symbol_strikes.size == 0:
            no_strikes_count += 1
            print(f"No strikes found for {symbol} with price {price} on {date}, row index: {index}")
            return np.nan  # Return NaN if no strikes are found for the symbol
        # Calculate the closest strike
        nearest_strike = symbol_strikes[np.abs(symbol_strikes - price).argmin()]
        processed_count += 1
        return nearest_strike

# Process each date column for last prices
for date in data_at.columns[3:]:  # Adjust index if your dates start from another column
    atm_strikes = []
    # Iterate over each row in the DataFrame
    for index, row in data_at.iterrows():
        symbol = row['Symbol']
        price = row[date]
        # Find the nearest strike for the symbol and price
        nearest_strike = find_nearest_strike(symbol, price, date, index)
        atm_strikes.append(nearest_strike)

    # Add the nearest strikes as a new column
    data_at[f'ATM {date}'] = atm_strikes

# Save the updated DataFrame to a new CSV file
data_at.to_csv(f'{BIN_F}/ATM(PREV_DAY_CLOSE_NOT_INDEXED).csv', index=False)

# Print the counts
print(f"Missing data count: {missing_count}")
print(f"Processed data count: {processed_count}")
print(f"No strikes available count: {no_strikes_count}")


Missing data count: 0
Processed data count: 7
No strikes available count: 0


In [ ]:
import pandas as pd
import os

def extract_dates_from_csv(directory):
    dates = []

    # Iterate through all files in the directory
    for filename in os.listdir(directory):
        if filename.endswith('.csv'):
            # Construct the full file path
            file_path = os.path.join(directory, filename)

            # Read the CSV file
            df = pd.read_csv(file_path)

            # Convert the 'Timestamp' column to datetime and extract the date part
            df['Timestamp'] = pd.to_datetime(df['Timestamp'])
            dates.extend(df['Timestamp'].dt.date.astype(str).tolist())

    # Remove duplicates and sort the dates
    unique_dates = sorted(set(dates))

    return unique_dates

# Example usage
directory = '/content/DATA1'
dates_list = extract_dates_from_csv(directory)
print(dates_list)


['2024-07-15', '2024-07-16', '2024-07-18', '2024-07-19', '2024-07-22', '2024-07-23', '2024-07-24']


In [ ]:
dates_list[0]

'2024-07-15'

In [ ]:
import pandas as pd

# Read CSV File
df = pd.read_csv(f'{BIN_F}/ATM(PREV_DAY_CLOSE_NOT_INDEXED).csv')


# Specify Column Index Name for Deletion
delete_index = "ATM "+ dates_list[0]

# Create New Column Headers
delete_index_position = df.columns.get_loc(delete_index)
new_columns = list(df.columns[:delete_index_position]) + list(df.columns[delete_index_position+1:]) + ['']

# Assign New Column Headers to DataFrame
df.columns = new_columns

# Drop Last Column
df = df.drop(df.columns[-1], axis=1)

# Save DataFrame to New CSV File
df.to_csv(f'{LOG_F}/ATM(PREV_DAY_CLOSE).csv', index=False)

print("New CSV file has been created successfully.")


New CSV file has been created successfully.


In [ ]:
print(dates_list)

['2024-07-15', '2024-07-16', '2024-07-18', '2024-07-19', '2024-07-22', '2024-07-23', '2024-07-24']


In [ ]:
print(date1)

2024-07-15


# ATM BASED ON CURR DAY OPEN (COMP)

In [ ]:
#Closig price for eveery day
import pandas as pd
import os

# Path to the directory containing the index files
data_directory = '/content/DATA1/'

# Load Index_FO data
index_fo_path = 'Comp_FO.csv'
index_fo_data = pd.read_csv(index_fo_path)

# Assuming 'Underlying Asset' column provides the filenames directly related to symbol CSVs
# Generate full file paths and check existence
index_fo_data['File Path'] = index_fo_data['Underlying Asset'].apply(lambda x: os.path.join(data_directory, f"{x}.csv"))

# Prepare to append data
all_dates = set()

# Iterate through each row in Index_FO to find corresponding index data and append closing prices
for idx, row in index_fo_data.iterrows():
    file_path = row['File Path']
    if os.path.exists(file_path):
        # Load the data from file
        data = pd.read_csv(file_path)
        data['Timestamp'] = pd.to_datetime(data['Timestamp']).dt.date  # Ensure date format

        # Extract and append unique dates
        unique_dates = data['Timestamp'].unique()
        all_dates.update(unique_dates)

        # Append closing prices
        for date in unique_dates:
            # Ensure the column for the date
            if date not in index_fo_data.columns:
                index_fo_data[date] = pd.NA  # Initialize with 'pd.NA' for missing data

            # Get the closing price for the date
            close_price = data.loc[data['Timestamp'] == date, 'Open'].iloc[0]  # Ensure correct extraction
            index_fo_data.at[idx, date] = close_price

# Remove the 'File Path' column from the DataFrame
index_fo_data.drop('File Path', axis=1, inplace=True)

# Save the modified Index_FO data
output_path = f'{BIN_F}/OPEN_PRICE.csv'
index_fo_data.to_csv(output_path, index=False)

# Print to confirm output
print(index_fo_data.head())


   Index     Symbol     Underlying Asset 2024-07-15 2024-07-16 2024-07-18  \
0     15  BANKNIFTY  NSE:NIFTYBANK-INDEX   52330.05    52466.7   52215.05   

  2024-07-19 2024-07-22 2024-07-23 2024-07-24  
0   52531.55    52145.6    52511.0   51657.65  


In [ ]:
# Load daily prices data and print the first few rows to inspect
import pandas as pd
daily_prices_path = f'{BIN_F}/OPEN_PRICE.csv'
daily_prices_data = pd.read_csv(daily_prices_path)
print("Daily Prices Data Sample:")
print(daily_prices_data.head())

# Load broker data, set column names, and print the first few rows to inspect
broker_data_path = NSE_FO_PATH_DRIVE
broker_data = pd.read_csv(broker_data_path, header=None)
column_names = [
    'ID', 'Description', 'Unknown1', 'Lot Size', 'Unknown2', 'Unknown3', 'Trading Window',
    'Date', 'Timestamp', 'Code', 'Unknown4', 'Unknown5', 'Unknown6', 'Index',
    'Unknown7', 'Strike Price', 'Option Type', 'Unknown8', 'Unknown9', 'Unknown10', 'Extra Column'
]
broker_data.columns = column_names
print("Broker Data Sample:")
print(broker_data.head())


Daily Prices Data Sample:
   Index     Symbol     Underlying Asset  2024-07-15  2024-07-16  2024-07-18  \
0     15  BANKNIFTY  NSE:NIFTYBANK-INDEX    52330.05     52466.7    52215.05   

   2024-07-19  2024-07-22  2024-07-23  2024-07-24  
0    52531.55     52145.6     52511.0    51657.65  
Broker Data Sample:
                ID                    Description  Unknown1  Lot Size  \
0  101124070835034  MIDCPNIFTY 24 Jul 08 14575 CE        14        75   
1  101124070835035  MIDCPNIFTY 24 Jul 08 14575 PE        14        75   
2  101124070835036  MIDCPNIFTY 24 Jul 08 14625 CE        14        75   
3  101124070835037  MIDCPNIFTY 24 Jul 08 14625 PE        14        75   
4  101124070835038  MIDCPNIFTY 24 Jul 08 14650 CE        14        75   

   Unknown2  Unknown3        Trading Window        Date   Timestamp  \
0      0.05       NaN  0915-1530|1815-1915:  2024-07-05  1720432800   
1      0.05       NaN  0915-1530|1815-1915:  2024-07-05  1720432800   
2      0.05       NaN  0915-1530|1815

In [ ]:
# Load daily prices data and print the first few rows to inspect
import pandas as pd
daily_prices_path = f'{BIN_F}/OPEN_PRICE.csv'
daily_prices_data = pd.read_csv(daily_prices_path)
print("Daily Prices Data Sample:")
print(daily_prices_data.head())

# Load broker data, set column names, and print the first few rows to inspect
broker_data_path = NSE_FO_PATH_DRIVE
broker_data = pd.read_csv(broker_data_path, header=None)
column_names = [
    'ID', 'Description', 'Unknown1', 'Lot Size', 'Unknown2', 'Unknown3', 'Trading Window',
    'Date', 'Timestamp', 'Code', 'Unknown4', 'Unknown5', 'Unknown6', 'Index',
    'Unknown7', 'Strike Price', 'Option Type', 'Unknown8', 'Unknown9', 'Unknown10', 'Extra Column'
]
broker_data.columns = column_names
print("Broker Data Sample:")
print(broker_data.head())


Daily Prices Data Sample:
   Index     Symbol     Underlying Asset  2024-07-15  2024-07-16  2024-07-18  \
0     15  BANKNIFTY  NSE:NIFTYBANK-INDEX    52330.05     52466.7    52215.05   

   2024-07-19  2024-07-22  2024-07-23  2024-07-24  
0    52531.55     52145.6     52511.0    51657.65  
Broker Data Sample:
                ID                    Description  Unknown1  Lot Size  \
0  101124070835034  MIDCPNIFTY 24 Jul 08 14575 CE        14        75   
1  101124070835035  MIDCPNIFTY 24 Jul 08 14575 PE        14        75   
2  101124070835036  MIDCPNIFTY 24 Jul 08 14625 CE        14        75   
3  101124070835037  MIDCPNIFTY 24 Jul 08 14625 PE        14        75   
4  101124070835038  MIDCPNIFTY 24 Jul 08 14650 CE        14        75   

   Unknown2  Unknown3        Trading Window        Date   Timestamp  \
0      0.05       NaN  0915-1530|1815-1915:  2024-07-05  1720432800   
1      0.05       NaN  0915-1530|1815-1915:  2024-07-05  1720432800   
2      0.05       NaN  0915-1530|1815

In [ ]:
import csv
import re

# Global variable for the search pattern

# Open the input CSV file
with open(NSE_FO_PATH_DRIVE, newline='') as csvfile:
    # Create a CSV reader object
    reader = csv.reader(csvfile)

    # Open the output CSV file to write the filtered data
    with open(f'{BIN_F}/broker.csv', 'w', newline='') as output_csvfile:
        # Create a CSV writer object
        writer = csv.writer(output_csvfile)

        # Iterate through each row in the input CSV file
        for row in reader:
            # Check if the first column contains the search pattern using regex
            if re.search(VALID_DATE, row[1]):
                # Append the first column and corresponding 15th column to the output CSV file
                writer.writerow([row[1], row[15]])


In [ ]:
import pandas as pd
df=pd.read_csv(f'{BIN_F}/broker.csv')
df.head(10)
df.tail(10)

,MIDCPNIFTY 24 Jul 08 14575 CE,14575.0
34938,BANKNIFTY 24 Jul 31 60100 CE,60100.0
34939,BANKNIFTY 24 Jul 31 60100 PE,60100.0
34940,BANKNIFTY 24 Jul 31 60200 CE,60200.0
34941,BANKNIFTY 24 Jul 31 60200 PE,60200.0
34942,BANKNIFTY 24 Jul 31 60300 CE,60300.0
34943,BANKNIFTY 24 Jul 31 60300 PE,60300.0
34944,BANKNIFTY 24 Jul 31 60400 CE,60400.0
34945,BANKNIFTY 24 Jul 31 60400 PE,60400.0
34946,BANKNIFTY 24 Jul 31 74000 CE,74000.0
34947,BANKNIFTY 24 Jul 31 74000 PE,74000.0


In [ ]:
import pandas as pd
import numpy as np

# Load the data from the specified CSV file
data_at = pd.read_csv(f'{BIN_F}/OPEN_PRICE.csv')
data_broker = pd.read_csv(f'{BIN_F}/broker.csv', header=None, names=['Description', 'Strike'])

# Ensure the 'Strike' column is treated as float
data_broker['Strike'] = pd.to_numeric(data_broker['Strike'], errors='coerce')
data_broker.dropna(subset=['Strike'], inplace=True)  # Drop any rows where 'Strike' could not be converted

# Initialize counters
missing_count = 0
processed_count = 0
no_strikes_count = 0

# Function to extract the symbol and find the closest strike
def find_nearest_strike(symbol, price, date, index):
    global missing_count, processed_count, no_strikes_count
    if pd.isna(price):
        missing_count += 1
        print(f"Missing price for {symbol} on {date}, row index: {index}")
        return np.nan  # Return NaN immediately if the price is missing
    else:
        price = float(price)  # Ensure price is a float to perform arithmetic operations
        filtered_data = data_broker[data_broker['Description'].str.contains(symbol, na=False)]
        symbol_strikes = filtered_data['Strike'].values
        if symbol_strikes.size == 0:
            no_strikes_count += 1
            print(f"No strikes found for {symbol} with price {price} on {date}, row index: {index}")
            return np.nan  # Return NaN if no strikes are found for the symbol
        # Calculate the closest strike
        nearest_strike = symbol_strikes[np.abs(symbol_strikes - price).argmin()]
        processed_count += 1
        return nearest_strike

# Process each date column for last prices
for date in data_at.columns[3:]:  # Adjust index if your dates start from another column
    atm_strikes = []
    # Iterate over each row in the DataFrame
    for index, row in data_at.iterrows():
        symbol = row['Symbol']
        price = row[date]
        # Find the nearest strike for the symbol and price
        nearest_strike = find_nearest_strike(symbol, price, date, index)
        atm_strikes.append(nearest_strike)

    # Add the nearest strikes as a new column
    data_at[f'ATM {date}'] = atm_strikes

# Save the updated DataFrame to a new CSV file
data_at.to_csv(f'{LOG_F}/ATM(CURR_DAY_OPEN).csv', index=False)

# Print the counts
print(f"Missing data count: {missing_count}")
print(f"Processed data count: {processed_count}")
print(f"No strikes available count: {no_strikes_count}")


Missing data count: 0
Processed data count: 7
No strikes available count: 0


# MERGING HL(REV HL,TICKER) + ATM(CURR , PREV) , SHIFTING LOG_F TO DATA DIR

In [ ]:
import pandas as pd

# Load the CSV files
file1 = f'{LOG_F}/ATM(CURR_DAY_OPEN).csv'
file2 =f'{LOG_F}/EVERYDAY_HL+REV_TICKERS.csv'



In [ ]:
import pandas as pd

# Load the CSV files

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# Merge the dataframes on the common columns
merged_df = pd.merge(df1, df2, on=["Index",  "Symbol", "Underlying Asset"], suffixes=('_HL', ''))

# Create a list of columns to select: Index, Name, Symbol, Underlying Asset, and columns matching 'ATM' and 'ATM Strike on [date] HL'
columns_to_select = ['Index',  'Symbol', 'Underlying Asset']
columns_to_select += [col for col in merged_df.columns if 'ATM' in col or 'ATM Strike on' in col]

# Filter the merged dataframe to include only the selected columns
final_df = merged_df[columns_to_select]

# Handle NaN values if necessary, for example, by filling them with an appropriate value or method
final_df = final_df.fillna('')

# Save the resulting dataframe to a new CSV file
output_file = f'{LOG_F}/ATM_CURR+REV_TICKERS.csv'
final_df.to_csv(output_file, index=False)

print(f"New file created: {output_file}")


New file created: /content/LOG_F/ATM_CURR+REV_TICKERS.csv


In [ ]:
#PREV_DAY CLOSE ATM  + REV TICKERS

In [ ]:
import pandas as pd

# Load the CSV files
file3 = f'{LOG_F}/ATM(PREV_DAY_CLOSE).csv'
file4 = f'{LOG_F}/EVERYDAY_HL+REV_TICKERS.csv'



In [ ]:
import pandas as pd

# Load the CSV files

df1 = pd.read_csv(file3)
df2 = pd.read_csv(file4)

# Merge the dataframes on the common columns
merged_df = pd.merge(df1, df2, on=["Index",  "Symbol", "Underlying Asset"], suffixes=('_HL', ''))

# Create a list of columns to select: Index, Name, Symbol, Underlying Asset, and columns matching 'ATM' and 'ATM Strike on [date] HL'
columns_to_select = ['Index',  'Symbol', 'Underlying Asset']
columns_to_select += [col for col in merged_df.columns if 'ATM' in col or 'ATM Strike on' in col]

# Filter the merged dataframe to include only the selected columns
final_df = merged_df[columns_to_select]

# Handle NaN values if necessary, for example, by filling them with an appropriate value or method
final_df = final_df.fillna('')

# Save the resulting dataframe to a new CSV file
output_file = f'{LOG_F}/ATM_PREV+REV_TICKERS.csv'
final_df.to_csv(output_file, index=False)

print(f"New file created: {output_file}")


New file created: /content/LOG_F/ATM_PREV+REV_TICKERS.csv


In [ ]:
import os
import shutil

source_folder = "/content/LOG_F"

# Get the list of folders in the base directory
folders = [f for f in os.listdir(jack) if os.path.isdir(os.path.join(jack, f))]

# Check if there is at least one folder in the base directory
if folders:
    first_folder = folders[0]
    destination_folder = os.path.join(jack, first_folder, os.path.basename(source_folder))

    # Copy the source folder to the destination folder
    shutil.copytree(source_folder, destination_folder, dirs_exist_ok=True)
    print(f"Copied {source_folder} to {destination_folder}")
else:
    print("No folders found in the base directory.")


Copied /content/LOG_F to /content/BNF/JUL_24/LOG_F


# DOWNLOADING

In [ ]:
import shutil
from google.colab import files

# Path to the folder you want to download
folder_path ='/content/NF/JUN_6'

# Path for the output zip file (make sure the .zip extension is included)
zip_path = 'JUN_6.zip'

# Create a zip file from the folder
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', folder_path)

# Download the zip file
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# FIL AND GRAPH

In [ ]:
#PLS CANGE THE BASE PATH EVERY TIME

The folder /content/FIL2 has been deleted.


In [ ]:
import os
import shutil
import pandas as pd
import ast

# Load the ATM+TICKERS2.csv
at_data = pd.read_csv('/content/LOG_F/ATM_CURR+REV_TICKERS.csv')

# Base and target paths
base_path = '/content/BNF/JUL_24/JUL_5S_24/'
target_base_path = '/content/Filtered'

# Fixed year-month part of the file name

# Function to create directories if they don't exist
def create_dirs(path):
    if not os.path.exists(path):
        os.makedirs(path)
    print(f"Directory created or verified: {path}")

# Iterate over rows and columns in the dataframe
for index, row in at_data.iterrows():
    symbol = row['Symbol']  # For example, 'NIFTY'
    for col in at_data.columns:
        if 'ATM Strike on ' in col:  # Check for ATM Info in column names
            date = col.split(' ')[3]  # Assuming the date is the fourth part of the column name
            atm_info = ast.literal_eval(row[col])
            ce_strikes = atm_info[0]
            pe_strikes = atm_info[1]

            # Create target directories for CE and PE
            ce_path = os.path.join(target_base_path, date, symbol, 'CE')
            pe_path = os.path.join(target_base_path, date, symbol, 'PE')
            create_dirs(ce_path)
            create_dirs(pe_path)

            # Process CE files
            for strike in ce_strikes:
                file_name = f"NSE:{symbol}{month_year}{int(strike)}CE.csv"
                source_file = os.path.join(base_path, 'CE', file_name)
                if os.path.exists(source_file):
                    shutil.copy(source_file, ce_path)
                    #print(f"File copied: {source_file} to {ce_path}")

            # Process PE files
            for strike in pe_strikes:
                file_name = f"NSE:{symbol}{month_year}{int(strike)}PE.csv"
                source_file = os.path.join(base_path, 'PE', file_name)
                if os.path.exists(source_file):
                    shutil.copy(source_file, pe_path)
                    #print(f"File copied: {source_file} to {pe_path}")


Directory created or verified: /content/Filtered/2024-07-15/BANKNIFTY/CE
Directory created or verified: /content/Filtered/2024-07-15/BANKNIFTY/PE
Directory created or verified: /content/Filtered/2024-07-16/BANKNIFTY/CE
Directory created or verified: /content/Filtered/2024-07-16/BANKNIFTY/PE
Directory created or verified: /content/Filtered/2024-07-18/BANKNIFTY/CE
Directory created or verified: /content/Filtered/2024-07-18/BANKNIFTY/PE
Directory created or verified: /content/Filtered/2024-07-19/BANKNIFTY/CE
Directory created or verified: /content/Filtered/2024-07-19/BANKNIFTY/PE
Directory created or verified: /content/Filtered/2024-07-22/BANKNIFTY/CE
Directory created or verified: /content/Filtered/2024-07-22/BANKNIFTY/PE
Directory created or verified: /content/Filtered/2024-07-23/BANKNIFTY/CE
Directory created or verified: /content/Filtered/2024-07-23/BANKNIFTY/PE
Directory created or verified: /content/Filtered/2024-07-24/BANKNIFTY/CE
Directory created or verified: /content/Filtered/20

In [ ]:
import os
import pandas as pd

# Define the base directory where the folders are stored
base_dir = '/content/Filtered'

# Define the output directory for the filtered data
output_dir = '/content/FIL2'

# Number of previous and future trading days to include
days_back = 1
days_front = 0

# Build a list of trading days from folder names
trading_days = sorted([pd.to_datetime(name).date() for name in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, name))])

# Function to get trading days range
def get_trading_days_range(current_day, trading_days, days_back, days_front):
    current_index = trading_days.index(current_day)
    # Get up to days_back previous days, if available
    start_index = max(0, current_index - days_back)
    # Get up to days_front future days, if available
    end_index = min(len(trading_days), current_index + days_front + 1)
    return trading_days[start_index:end_index]

# Ensure the output directory exists
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Iterate over each folder in the base directory
for folder_name in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder_name)
    if os.path.isdir(folder_path):
        # Convert the folder name to a date object
        folder_date = pd.to_datetime(folder_name).date()
        dates_range = get_trading_days_range(folder_date, trading_days, days_back, days_front)

        # Collect dates in a list
        dates_to_include = dates_range

        # Iterate over all subdirectories and files within the folder
        for sub_dir in os.listdir(folder_path):
            sub_dir_path = os.path.join(folder_path, sub_dir)
            if os.path.isdir(sub_dir_path):
                for ce_pe_dir in os.listdir(sub_dir_path):
                    ce_pe_dir_path = os.path.join(sub_dir_path, ce_pe_dir)
                    if os.path.isdir(ce_pe_dir_path):
                        for file in os.listdir(ce_pe_dir_path):
                            file_path = os.path.join(ce_pe_dir_path, file)
                            if file_path.endswith('.csv'):
                                # Load the CSV
                                data = pd.read_csv(file_path)
                                # Convert 'Timestamp' to datetime format
                                data['Timestamp'] = pd.to_datetime(data['Timestamp'])
                                # Filter the data to include only the specified dates
                                filtered_data = data[data['Timestamp'].dt.date.isin(dates_to_include)]

                                # Construct the output file path
                                output_file_path = os.path.join(output_dir, folder_name, sub_dir, ce_pe_dir, file)
                                # Ensure the output file directory exists
                                os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
                                # Save the filtered data to the new file
                                filtered_data.to_csv(output_file_path, index=False)

print("Data filtering complete.")


Data filtering complete.


In [ ]:
#PLOTTING

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import re
import os

def load_and_prepare_data(file_path):
    """Load CSV data and preprocess."""
    df = pd.read_csv(file_path)
    if df.empty:
        return df  # Return empty DataFrame
    df['Timestamp'] = pd.to_datetime(df['Timestamp'])
    df['Minutes'] = df['Timestamp'].dt.hour * 60 + df['Timestamp'].dt.minute
    return df

def calculate_day_indices(df):
    """Calculate the start and end indices of each trading day."""
    first_indices = df.groupby(df['Timestamp'].dt.date).head(1).index.tolist()
    last_indices = df.groupby(df['Timestamp'].dt.date).tail(1).index.tolist()
    return first_indices, last_indices

def adjust_minutes_for_continuity(df, first_indices, last_indices):
    """Adjust minutes for visual continuity in plotting."""
    df['Minutes'] = (df['Timestamp'] - df['Timestamp'].iloc[0]).dt.total_seconds() / 60
    for i in range(len(first_indices) - 1):
        day_gap = df.loc[first_indices[i + 1], 'Minutes'] - df.loc[last_indices[i], 'Minutes'] - 1
        df.loc[first_indices[i + 1]:, 'Minutes'] -= day_gap
    return df

def load_atm_data(atm_file_path):
    """Load ATM CSV data and preprocess."""
    atm_df = pd.read_csv(atm_file_path)
    # Reshape the DataFrame to get the correct format
    date_columns = [col for col in atm_df.columns if re.match(r'ATM \d{4}-\d{2}-\d{2}', col)]
    atm_df = atm_df.melt(id_vars=['Symbol'], value_vars=date_columns, var_name='Date', value_name='ATM')
    atm_df['Date'] = atm_df['Date'].str.extract(r'(\d{4}-\d{2}-\d{2})')[0]
    atm_df['Date'] = pd.to_datetime(atm_df['Date'])
    atm_df = atm_df.set_index(['Symbol', 'Date'])
    return atm_df

def plot_data(df, first_indices, last_indices, symbol, date, atm_values, file_path, plot_size=(16, 8), shift_1530=0):
    """Plot the trading data with adjusted minutes and specific time labels for key trading times."""
    fig, ax1 = plt.subplots(figsize=plot_size)

    ax1.set_xlabel('Time', fontsize=14)
    ax1.set_ylabel('Open', color='purple', fontsize=14)
    ax1.plot(df['Minutes'], df['Open'], label='Open', color='purple')
    ax1.tick_params(axis='y', labelcolor='purple', labelsize=12)
    ax1.tick_params(axis='x', labelsize=12)

    ax2 = ax1.twinx()
    ax2.set_ylabel('Volume', color='blue', fontsize=14)
    ax2.plot(df['Minutes'], df['Volume'], label='Volume', color='blue', alpha=0.6)
    ax2.tick_params(axis='y', labelcolor='blue', labelsize=12)

    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    ax3.set_ylabel('OI', color='green', fontsize=14)
    ax3.plot(df['Minutes'], df['OI'], label='OI', color='orange', alpha=0.7)
    ax3.tick_params(axis='y', labelcolor='green', labelsize=12)

    # Add vertical dashed lines at the first element of each trading day
    for idx in first_indices:
        first_minute = df.loc[idx, 'Minutes']
        ax1.axvline(x=first_minute, color='black', linestyle='--', linewidth=1)

    # Set x-axis ticks to time labels
    x_ticks = []
    x_labels = []
    key_times = [9 * 60 + 15, 9 * 60 + 30, 10 * 60 + 30, 11 * 60 + 30, 12 * 60 + 30, 13 * 60 + 30, 14 * 60 + 30]  # Minutes after 9:00 AM

    for idx, end_idx in zip(first_indices, last_indices):
        day_start_minute = df.loc[idx, 'Minutes']
        day_start_hour = df.loc[idx, 'Timestamp'].hour
        day_start_minute_actual = df.loc[idx, 'Timestamp'].minute

        # Calculate and append tick values and labels for each key time
        for kt in key_times:
            adjusted_minute = day_start_minute + (kt - day_start_hour * 60 - day_start_minute_actual)
            x_ticks.append(adjusted_minute)
            time = df.loc[idx, 'Timestamp'].replace(hour=kt // 60, minute=kt % 60)
            month_str = time.strftime('%b')
            if kt == 9 * 60 + 15:  # Special case for 9:15 with extended label formatting
                label = f"{time.strftime('%H')}\n{time.strftime('%M')}\n\n{time.strftime('%d')} {month_str}"
            else:
                label = f"{time.strftime('%H')}\n{time.strftime('%M')}"
            x_labels.append(label)

        # Add the last timestamp of the day labeled as 15:30, apply manual shift if needed
        last_minute = df.loc[end_idx, 'Minutes'] + shift_1530
        x_ticks.append(last_minute)
        last_time = df.loc[end_idx, 'Timestamp'].replace(hour=15, minute=30)
        month_str = last_time.strftime('%b')
        x_labels.append(f"{last_time.strftime('%H')}\n{last_time.strftime('%M')}")

    # Add vertical dashed lines for each specified timestamp
    for x in x_ticks:
        ax1.axvline(x=x, color='black', linestyle='--', linewidth=1)

    # Get the ATM value
    try:
        key = (symbol, pd.to_datetime(date))
        atm_value = atm_values.loc[key, 'ATM']
    except KeyError:
        atm_value = 'N/A'

    # Split the data by date
    dates = df['Timestamp'].dt.date.unique()
    if len(dates) > 1:
        current_day = df[df['Timestamp'].dt.date == dates[-1]]
        previous_day = df[df['Timestamp'].dt.date == dates[-2]]

        # Additional volume information for each day
        first_volume_current = current_day['Volume'].iloc[0]
        last_volume_current = current_day['Volume'].iloc[-1]
        avg_volume_current = current_day['Volume'].mean()
        std_volume_current = current_day['Volume'].std()

        first_volume_previous = previous_day['Volume'].iloc[0]
        last_volume_previous = previous_day['Volume'].iloc[-1]
        avg_volume_previous = previous_day['Volume'].mean()
        std_volume_previous = previous_day['Volume'].std()

        # Title with the specified features
        title = (
            f'{file_path}\n'
            f'Current Day - First Vol: {first_volume_current}, Last Vol: {last_volume_current}, '
            f'Avg Vol: {avg_volume_current:.2f}, Std Vol: {std_volume_current:.2f}\n'
            f'Previous Day - First Vol: {first_volume_previous}, Last Vol: {last_volume_previous}, '
            f'Avg Vol: {avg_volume_previous:.2f}, Std Vol: {std_volume_previous:.2f}, ATM: {atm_value}'
        )
    else:
        # Only one day's data present
        first_volume = df['Volume'].iloc[0]
        last_volume = df['Volume'].iloc[-1]
        avg_volume = df['Volume'].mean()
        std_volume = df['Volume'].std()

        title = (
            f'{file_path} (First Vol: {first_volume}, '
            f'Last Vol: {last_volume}, Avg Vol: {avg_volume:.2f}, '
            f'Std Vol: {std_volume:.2f}, ATM: {atm_value})'
        )

    ax1.set_title(title, fontsize=12)

    plt.xticks(ticks=x_ticks, labels=x_labels, fontsize=12)

    fig.tight_layout()
    plt.grid(True)
    plt.show()

def process_directory(base_dir, atm_values):
    """Process all CSV files in the directory structure."""
    for root, dirs, files in os.walk(base_dir):
        # Sort directories and files to ensure sequential processing
        dirs.sort()
        files.sort()
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                # Extract the date and symbol from the directory structure
                try:
                    date_str = os.path.basename(os.path.dirname(os.path.dirname(root)))
                    symbol = os.path.basename(os.path.dirname(root))
                    date = pd.to_datetime(date_str).date()
                except ValueError:
                    continue

                try:
                    df = load_and_prepare_data(file_path)
                    if df.empty:
                        continue
                    first_indices, last_indices = calculate_day_indices(df)
                    df = adjust_minutes_for_continuity(df, first_indices, last_indices)
                    plot_data(df, first_indices, last_indices, symbol, date, atm_values, file_path, plot_size=(16, 8), shift_1530=-11)
                except Exception as e:
                    print(f"Error processing {file_path}: {e}")
                    continue

# Load ATM data
atm_file_path = '/content/LOG_F/ATM_CURR+REV_TICKERS.csv'  # Update this path to your actual ATM CSV file location
atm_values = load_atm_data(atm_file_path)

# Process directory with the loaded ATM values
process_directory('/content/FIL2', atm_values)





# Copy To Drive

In [ ]:
import shutil
import os

# Define source and destination directories
source_dir = '/content/BNF'
destination_dir = '/content/drive/MyDrive/BT_DATA'

# Ensure the destination directory exists
os.makedirs(destination_dir, exist_ok=True)

# Copy the contents of the source directory to the destination directory
shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)

print("Copying completed successfully.")


Copying completed successfully.


# PLATINUM
